# Hadamard Gate

Example of using the Logical Assembler to generate a circuit containing a logical
Hadamard.

In [ ]:
# This file contains information which is proprietary to Riverlane Ltd
# ("Riverlane") and is Riverlane Confidential Information.
# (c) Copyright Riverlane 2025-2026. All rights reserved.
from deltakit_compile.frontend.logasm import LogAsmBuilder, LogAsmProgram, RotatedPlanarPatch

In [3]:
def build_hadamard(width: int, height: int) -> LogAsmProgram:
    """
    Build a LogASM program that implements a logical Hadamard gate.

    Args:
        width: The width of the patch.
        height: The height of the patch.

    Returns:
        Instantiated subroutine object to be provided to the LogicalAssembler.
    """
    builder = LogAsmBuilder()
    lq0 = builder.declare_patch(
        RotatedPlanarPatch(width=width, height=height, location=(0, 0), vertical_z=False)
    )
    stab_rounds = max(width, height)

    lq0.prepare("Z")
    lq0.measure_stabilisers(stab_rounds)

    # Apply the transversal hadamard gate, which implicitly rotates the patch
    lq0.transversal("H")

    # Rotate the patch back to its original orientation, changing its location in the process
    lq0.rotate(offset=(width, 0))
    lq0.measure("X")

    return builder.build_program()

In [4]:
hadamard_circuit = build_hadamard(3, 3)
print(hadamard_circuit)

LogAsmProgram({
  %qreg = log_asm.patch_dec -> !log_asm.patch.rot_planar<size=(3, 3), location=(0.0, 0.0), orient=h_z>
  %qreg_1 = log_asm.prepare<Z> (%qreg : !log_asm.patch.rot_planar<size=(3, 3), location=(0.0, 0.0), orient=h_z>)
  %qreg_2 = log_asm.meas_stab<3> (%qreg_1 : !log_asm.patch.rot_planar<size=(3, 3), location=(0.0, 0.0), orient=h_z>)
  %qreg_3 = log_asm.transversal<H> (%qreg_2 : !log_asm.patch.rot_planar<size=(3, 3), location=(0.0, 0.0), orient=h_z>) -> !log_asm.patch.rot_planar<size=(3, 3), location=(0.0, 0.0), orient=h_z>
  %qreg_4 = log_asm.rotate<3> (%qreg_3 : !log_asm.patch.rot_planar<size=(3, 3), location=(0.0, 0.0), orient=h_z>) -> !log_asm.patch.rot_planar<size=(3, 3), location=(3.0, 0.0), orient=v_z>
  %cexpr = log_asm.measure<X> (%qreg_4 : !log_asm.patch.rot_planar<size=(3, 3), location=(3.0, 0.0), orient=v_z>) -> i1
  qstruct.output(:)
})
